In [ ]:
pip install gradio openai Pillow

In [6]:
import gradio as gr
import base64
import io
import re
from openai import OpenAI
from PIL import Image
import random
from collections import Counter

# ---------------------------
# Replace with your NVIDIA API Key (or better: load from env)
# ---------------------------
client = OpenAI(
    api_key="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",
    base_url="https://integrate.api.nvidia.com/v1"
)

PRICE_LIST = {
    "apple": 30,
    "banana": 20,
    "orange": 25,
    "mango": 50,
    "grapes": 60,
    "watermelon": 80,
    "pineapple": 70,
    "papaya": 45
}

# Global list to store identified items
identified_items = []

def _safe_extract_message_content(response):
    """
    Try several common ways to get the text content from the model response object.
    """
    # If response is None, return empty string
    if not response:
        return ""

    # Many wrappers return: response.choices[0].message.content or response.choices[0].text
    try:
        return response.choices[0].message.content
    except Exception:
        pass
    try:
        return response.choices[0].message["content"]
    except Exception:
        pass
    try:
        return response.choices[0].text
    except Exception:
        pass
    # Fallback to string cast
    return str(response)


def _normalize_item_name(name):
    """
    Normalize detected name to match keys in PRICE_LIST.
    Strategy:
    - Lowercase
    - Remove quantity annotations like '(2)', 'x2', '2x'
    - Strip adjectives if name contains a known fruit word
    - If exact match not found, try to find a key that's substring of name
    """
    name = name.lower().strip()
    # Remove quantity-like parts
    name = re.sub(r"\(\s*\d+\s*\)", "", name)       # remove (2)
    name = re.sub(r"\bx ?\d+\b", "", name)          # remove x2 or x 2
    name = re.sub(r"\b\d+ ?x\b", "", name)          # remove 2x
    name = re.sub(r"[^a-z\s]", " ", name)           # drop punctuation
    name = re.sub(r"\s+", " ", name).strip()

    if not name:
        return None

    # exact match
    if name in PRICE_LIST:
        return name

    # singularize simple plural (apples -> apple)
    if name.endswith("s") and name[:-1] in PRICE_LIST:
        return name[:-1]

    # try substring matching: e.g. "green apple" -> "apple"
    for key in PRICE_LIST.keys():
        if key in name:
            return key

    # otherwise unknown
    return None


def identify_and_bill(image):
    global identified_items
    try:
        # Convert image to base64 data URL
        buffered = io.BytesIO()
        image.save(buffered, format="PNG")
        img_b64 = base64.b64encode(buffered.getvalue()).decode()

        # Build a robust messages prompt
        messages = [
            {"role": "system", "content": "You are a vision AI that identifies grocery items and fruits. Return only a comma-separated list of item names (no prices)."},
            {"role": "user", "content": [
                {"type": "text", "text": "Identify all fruits or grocery items in this image as comma-separated names. Use plain names like 'apple, banana, apple' if there are two apples."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
            ]}
        ]

        response = client.chat.completions.create(
            model="meta/llama-3.2-11b-vision-instruct",
            messages=messages,
            temperature=0.2,
        )

        # Extract text safely
        raw_text = _safe_extract_message_content(response)
        if raw_text is None:
            raw_text = ""

        raw_text = str(raw_text).strip()
        raw_text_lower = raw_text.lower()

        # If nothing returned, try fallback: maybe model returned JSON or lines
        if not raw_text_lower:
            return image, "⚠️ No items detected. Try a clearer photo."

        # Split into candidate tokens by commas or newlines
        tokens = re.split(r"[,\n]+", raw_text_lower)
        tokens = [t.strip() for t in tokens if t.strip()]

        # Further split tokens that may be 'apple 2' or 'apple x2' etc.
        expanded_tokens = []
        for t in tokens:
            # look for 'item number' patterns like 'apple 2' or 'apple (2)'
            m = re.search(r"(.+?)\s*(?:\(|x|times|x)\s*?(\d+)\)?$", t)
            if m:
                name_part = m.group(1).strip()
                qty = int(m.group(2))
                expanded_tokens.extend([name_part] * max(1, qty))
                continue

            # look for 'item - qty' or 'item: qty'
            m2 = re.search(r"(.+?)[:\-]\s*(\d+)$", t)
            if m2:
                expanded_tokens.extend([m2.group(1).strip()] * int(m2.group(2)))
                continue

            # otherwise just add token once
            expanded_tokens.append(t)

        # Normalize names and map to price keys
        normalized = []
        unknowns = []
        for t in expanded_tokens:
            k = _normalize_item_name(t)
            if k:
                normalized.append(k)
            else:
                unknowns.append(t)

        if not normalized:
            # If nothing matched known items, still try to present the raw model output
            return image, f"⚠️ Detected items but none matched known price list.\n\nModel output: `{raw_text}`"

        # Add newly identified items to the global list
        identified_items.extend(normalized)

        # Generate bill from accumulated items
        counts = Counter(identified_items)

        total = 0
        bill_lines = []
        for item_key, qty in counts.items():
            price_each = PRICE_LIST.get(item_key, random.randint(20, 100))
            line_total = price_each * qty
            total += line_total
            bill_lines.append(f"{item_key.title()} x{qty} = ₹{price_each} each → ₹{line_total}")

        bill_lines.append(f"\n💰 **Total = ₹{total}**")

        # If there were unknown tokens from the *current* image, include them as note
        if unknowns:
            bill_lines.append("\n⚠️ Unrecognized/ambiguous items (couldn't match price list) in this image:")
            bill_lines.append(", ".join(sorted(set(unknowns))))

        bill_text = "🧾 **Smart Retail Bill**\n\n" + "\n".join(bill_lines)
        return image, bill_text

    except Exception as e:
        return image, f"⚠️ Error: {str(e)}"


# ---------------------------
# Gradio Interface
# ---------------------------
demo = gr.Interface(
    fn=identify_and_bill,
    inputs=gr.Image(type="pil", label="Upload Fruit or Grocery Image"),
    outputs=[
        gr.Image(label="Uploaded Image"),
        gr.Markdown(label="Generated Bill")
    ],
    title="🛍️ Smart Grocery Billing and Object Detection System Using AI",
    description="Upload an image of fruits or groceries. The NVIDIA Vision AI identifies the items and generates a bill automatically.",
    theme="soft"
)

if __name__ == "__main__":
    demo.launch(debug=True, share=True, prevent_thread_lock=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ec9b6df452d175fc8d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1133, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py",

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e2a0b6774a1bb89b9b.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://ec9b6df452d175fc8d.gradio.live


In [ ]:
!pip install gradio openai pillow sqlite-utils
